# Ablation Study 1: Vision-Only CLIP Retrieval

This notebook evaluates the baseline retrieval performance of a vanilla vision-only CLIP pipeline on the DeepFashion In-Shop Clothes Retrieval Benchmark.

In this experiment, both query and gallery images are encoded using the pretrained CLIP ViT-B/32 image encoder without:
- fine-tuning,
- caption generation,
- or multimodal fusion.

Retrieval is performed using cosine similarity search with FAISS.

This study serves as the baseline configuration for comparing the impact of later enhancements such as contrastive fine-tuning and multimodal feature fusion.

## Evaluation Metrics

The system is evaluated using:
- Recall@K
- NDCG@K
- mAP@K

---

## Retrieval Pipeline

```text
Query Image
    ↓
CLIP Image Encoder
    ↓
Image Embedding
    ↓
FAISS Similarity Search
    ↓
Top-K Retrieved Images
```

## Install Dependencies and Imports

In [1]:
!pip install git+https://github.com/openai/CLIP.git -q
!pip install faiss-cpu -q
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import clip
import faiss

from PIL import Image
from torch.utils.data import Dataset, DataLoader

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.9 MB/s eta 0:00:00


## Configuration

In [2]:
ROOT = '/kaggle/input/datasets/dedeepyaavancha/deepfashion-in-shop-clothes-retrieval-benchmark/In-shop Clothes Retrieval Benchmark'

SAVE_DIR = '/kaggle/working/ablation_clip_only'
os.makedirs(SAVE_DIR, exist_ok=True)

SEEDS = [2023006, 2023008, 2023524, 2023552]

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

cuda


## Load Dataset

In [3]:
eval_path = os.path.join(ROOT, 'Eval/list_eval_partition.txt')
bbox_path = os.path.join(ROOT, 'Anno/list_bbox_inshop.txt')

df = pd.read_csv(
    eval_path,
    sep=r'\s+',
    skiprows=2,
    header=None,
    names=['image_name','item_id','split']
)

df_bbox = pd.read_csv(
    bbox_path,
    sep=r'\s+',
    skiprows=2,
    header=None,
    names=['image_name','clothes_type','pose_type','x1','y1','x2','y2']
)

df = df.merge(df_bbox, on='image_name')

df['image_path'] = df['image_name'].apply(
    lambda x: os.path.join(ROOT, x)
)

df_query = df[df['split']=='query'].reset_index(drop=True)
df_gallery = df[df['split']=='gallery'].reset_index(drop=True)

gallery_ids_arr = np.array(df_gallery['item_id'].tolist())
query_ids = df_query['item_id'].tolist()

print("Query:", len(df_query))
print("Gallery:", len(df_gallery))

Query: 14218
Gallery: 12612


## Load Model

In [4]:
clip_model, preprocess = clip.load(
    'ViT-B/32',
    device=device
)

clip_model.eval()

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 135MiB/s]


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

## Dataset Class

In [5]:
class FashionDataset(Dataset):

    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = Image.open(
            row['image_path']
        ).convert('RGB')

        return preprocess(img), row['item_id']

## Helper Functions

In [6]:
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def embed_split(df):

    loader = DataLoader(
        FashionDataset(df),
        batch_size=256,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    embs = []

    with torch.no_grad():

        for imgs, _ in loader:

            imgs = imgs.to(device)

            e = clip_model.encode_image(imgs)

            e = e / e.norm(dim=-1, keepdim=True)

            embs.append(e.cpu().numpy())

    return np.vstack(embs)

def run_retrieval(g, q):

    index = faiss.IndexFlatIP(g.shape[1])

    index.add(g.astype(np.float32))

    sims, idxs = index.search(
        q.astype(np.float32),
        15
    )

    return compute_metrics(idxs)

## Evaluation Metrics

In [7]:
def compute_metrics(indices, Ks=[5,10,15]):

    results = {}

    for K in Ks:

        recalls = []
        ndcgs = []
        aps = []

        for i, qid in enumerate(query_ids):

            retrieved = gallery_ids_arr[
                indices[i,:K]
            ]

            relevant = (retrieved == qid)

            recalls.append(
                1.0 if relevant.any() else 0.0
            )

            dcg = sum(
                rel / math.log2(idx+2)
                for idx, rel in enumerate(relevant)
            )

            idcg = sum(
                1.0 / math.log2(idx+2)
                for idx in range(
                    min(int(relevant.sum()), K)
                )
            )

            ndcgs.append(
                dcg/idcg if idcg>0 else 0
            )

            hits = 0
            prec = 0

            for idx, rel in enumerate(relevant):

                if rel:

                    hits += 1
                    prec += hits/(idx+1)

            aps.append(
                prec / max(1, int(relevant.sum()))
            )

        results[K] = {
            "Recall": np.mean(recalls),
            "NDCG": np.mean(ndcgs),
            "mAP": np.mean(aps)
        }

    return results

## Main Evaluation Loop

In [8]:
results = []

for seed in SEEDS:

    print(f"\n========== SEED {seed} ==========")

    set_seed(seed)

    g_img = embed_split(df_gallery)
    q_img = embed_split(df_query)

    metrics = run_retrieval(
        g_img,
        q_img
    )

    row = {"seed": seed}

    for K in [5,10,15]:

        row[f"R@{K}"] = metrics[K]["Recall"]
        row[f"NDCG@{K}"] = metrics[K]["NDCG"]
        row[f"mAP@{K}"] = metrics[K]["mAP"]

    results.append(row)

    print(metrics)


========== SEED 2023006 ==========
{5: {'Recall': np.float64(0.672316781544521), 'NDCG': np.float64(0.5627630448355583), 'mAP': np.float64(0.5167177560525781)}, 10: {'Recall': np.float64(0.74250949500633), 'NDCG': np.float64(0.5737849743742952), 'mAP': np.float64(0.4977163404227396)}, 15: {'Recall': np.float64(0.7814038542692362), 'NDCG': np.float64(0.5751733472423869), 'mAP': np.float64(0.48018130320438107)}}

========== SEED 2023008 ==========
{5: {'Recall': np.float64(0.672316781544521), 'NDCG': np.float64(0.5627630448355583), 'mAP': np.float64(0.5167177560525781)}, 10: {'Recall': np.float64(0.74250949500633), 'NDCG': np.float64(0.5737849743742952), 'mAP': np.float64(0.4977163404227396)}, 15: {'Recall': np.float64(0.7814038542692362), 'NDCG': np.float64(0.5751733472423869), 'mAP': np.float64(0.48018130320438107)}}

========== SEED 2023524 ==========
{5: {'Recall': np.float64(0.672316781544521), 'NDCG': np.float64(0.5627630448355583), 'mAP': np.float64(0.5167177560525781)}, 10: {'Re

## Final Results

In [9]:
df_res = pd.DataFrame(results)

mean_res = df_res.drop(columns=["seed"]).mean()
std_res = df_res.drop(columns=["seed"]).std()

print("\n===== FINAL RESULTS =====")

for col in mean_res.index:

    print(
        f"{col}: "
        f"{mean_res[col]:.4f} ± {std_res[col]:.4f}"
    )

df_res.to_csv(
    f"{SAVE_DIR}/ablation_1_results.csv",
    index=False
)

summary_df = pd.DataFrame({
    "metric": mean_res.index,
    "mean": mean_res.values,
    "std": std_res.values
})

summary_df.to_csv(
    f"{SAVE_DIR}/ablation_1_summary.csv",
    index=False
)

print("Saved results.")


===== FINAL RESULTS =====
R@5: 0.6723 ± 0.0000
NDCG@5: 0.5628 ± 0.0000
mAP@5: 0.5167 ± 0.0000
R@10: 0.7425 ± 0.0000
NDCG@10: 0.5738 ± 0.0000
mAP@10: 0.4977 ± 0.0000
R@15: 0.7814 ± 0.0000
NDCG@15: 0.5752 ± 0.0000
mAP@15: 0.4802 ± 0.0000
Saved results.
